# Crop data quality audit

Source CSV: Akshat Gupta, Kaggle v1, CC BY-SA 4.0. Results are descriptive arithmetic and coverage diagnostics. The original Power BI report is preserved and was not rendered or refreshed.


In [1]:
from pathlib import Path
import json
from etl.__main__ import audit
from etl.policy import Policy
from etl.extract import read_csv
from etl.clean import normalize
from etl.analysis import insights
root = Path.cwd()
policy = Policy.load(root / "configs/policy.json")
profile = audit(root / "data/raw/crop_yield.csv", root / "configs/policy.json", root / "data/processed")
rows = normalize(read_csv(root / "data/raw/crop_yield.csv", policy.data_sha256), policy)
metrics = insights(rows)


In [2]:
print(json.dumps({k: metrics[k] for k in ["rows", "crops", "state_labels", "seasons", "minimum_year", "maximum_year"]}, indent=2))


{
  "rows": 19689,
  "crops": 55,
  "state_labels": 30,
  "seasons": 6,
  "minimum_year": 1997,
  "maximum_year": 2020
}


The labels include states and UTs. Coverage varies; a 1997–2020 range is not balanced coverage of every crop/state.


In [3]:
print(json.dumps({k: metrics[k] for k in ["ratio_mismatches", "ratio_mismatch_percent", "ratio_check_pass"]}, indent=2))


{
  "ratio_mismatches": 9717,
  "ratio_mismatch_percent": "49.35",
  "ratio_check_pass": 9972
}


Reported yield is retained. The ratio uses source decimal area and a symmetric 5% relative OR 0.01 absolute tolerance. A mismatch does not establish the reported agronomic yield is wrong. Physical production/yield units are unverified per crop.


In [4]:
print(json.dumps({k: metrics[k] for k in ["fractional_area_rows", "telangana_before_2014_rows", "boundary_review_rows"]}, indent=2))


{
  "fractional_area_rows": 235,
  "telangana_before_2014_rows": 60,
  "boundary_review_rows": 141
}


The decoded original cache rounds 235 fractional areas to integers. The audit preserves them. State alias mapping changes names, not borders; split-year and pre-split Telangana rows are retained and flagged.


In [5]:
print(json.dumps({k: metrics[k] for k in ["row_weighted_rainfall_average", "row_weighted_pesticide_average"]}, indent=2))
comparison = json.loads((root / "reports/model/arithmetic-crosschecks.json").read_text())
print(json.dumps(comparison, indent=2))


{
  "row_weighted_rainfall_average": "1437.755176615968307176596069",
  "row_weighted_pesticide_average": "48848.35339200924374015947991"
}
{
  "cached_area_sum_diagnostic": 3542574244,
  "cached_rows": 19689,
  "count_nonblank_rows": 19689,
  "csv_area_sum_diagnostic": "3542574242.797",
  "csv_rows": 19689,
  "distinct_crop_labels_trimmed": 55,
  "distinct_seasons_trimmed": 6,
  "distinct_state_labels_trimmed": 30,
  "distinct_years": 24,
  "fractional_area_values_changed_in_cache": 235,
  "identical_label_grains": true,
  "limitations": "No DAX engine, report interaction, visual result or rendering was executed. Inline aliases may differ from display labels. Means are weighted by source rows, not a national weather/pesticide estimate. No cross-crop production or yield totals are interpreted.",
  "row_weighted_pesticide_average_cached": 48848.353392009245,
  "row_weighted_pesticide_average_csv": "48848.35339200924374015947991",
  "row_weighted_rainfall_average_cached": 1437.7551766159

These column means are weighted by crop-season rows, with repeated annual observations. They are not national weather or pesticide-use estimates. Cached pandas arithmetic is not validated against Power BI output. No cross-crop production totals or productivity ranking is interpreted. The hero is a Python re-creation, not a Power BI export.
